# 39 · Deep conditioning — EHR can thiệp **trong lúc** biểu diễn hình thành

## Câu hỏi, và vì sao nó vẫn mở

Backbone đông cứng ⇒ EHR chỉ **điều biến một biểu diễn đã xong**. Nó không tạo ra được đặc trưng
chưa tồn tại. `11` đo cùng một cơ chế ở hai vị trí:

```
adapter TRONG block 5,7      G_mid  +0.0036
late fusion tren patch              +0.0003      gap 10 lan
```

Ở block 8, `Z' = Z(1+γ) + β` còn đi qua **4 block attention nữa** — attention là hàm **phi tuyến**
của `Z`, nên cùng một `(γ, β)` toàn cục đổi được **patch nào chú ý tới patch nào**. Ở đầu ra thì
không: nó chỉ co giãn một vector đã xong. **Cùng phép toán, hai lớp hàm khác hẳn.**

## 5 arm — tách **VỊ TRÍ** khỏi **HÌNH DẠNG**

Chạy FiLM một mình thì kết quả **âm** không phân biệt được *"vị trí không quan trọng"* với
*"FiLM sai hình dạng"*. Nên chạy **cả hai hình dạng ở CÙNG vị trí**, mỗi cái một đối chứng khớp
tham số:

```
D1      KHONG adapter                    <- gan nhu mien phi (block 9-12 chay no_grad)
D2_U    FiLM   adapter, dau vao HANG SO
D2_E    FiLM   adapter, dau vao EHR
D3_U    XAttn  adapter, dau vao HANG SO
D3_E    XAttn  adapter, dau vao EHR

G_film  = D2_E - D2_U        G_xattn = D3_E - D3_U
G_xattn - G_film             HINH DANG co quan trong khong, o CUNG vi tri
capacity = D2_U - D1         phan CHI do them tham so
```

**Chỉ `G_*` được đọc.** `06b` chứng minh cặp này cứu kết luận: `H6z 0.4330` vs `H6 0.4339` cho thấy
token EHR không thêm gì, trong khi so với `H5 0.4336` thì **trông như có**. `52b` vừa lặp lại đúng
bài học đó: `t1_col_zero` **thắng** `t1_col` `+0.0015`.

## Prior đã yếu đi — phải nói trước

`52b` đo `ColTokBag` **không mang nội dung** ở tầng ngoài (`t1_col − t1_col_zero = −0.0015`), dù
`Tier1` chọn lọc rất mạnh (entropy `29.3%` so với đều, `sd/BN` gấp `9.03×` đối chứng hằng số). Đưa
**cùng bộ mã hoá đó** vào sâu hơn chưa chắc cứu được. Và `+0.0036` của `11` đo trên **cohort cũ, 13
nhãn, đúng một nhãn tail** — không tự động chuyển sang 40 nhãn với 11 nhãn tail.

Nên `EHR_SRC` là **cờ**: `nb53` cho biết nút thắt có phải bộ mã hoá không, rồi mới chọn `col` hay `xgb`.

## C0 · Config

In [ ]:
import os, json, math, time, tarfile, io as _io, re, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'r1'
RES39 = os.path.join(OUT_DIR, 'stage4_nb39', RUN_TAG); CKPT = os.path.join(RES39, 'ckpt')
P_OOF = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
P_TOUCH = os.path.join(OUT_DIR, '_test_touches.jsonl')

def ensure_res():
    '''Drive FUSE danh roi thu muc RONG — goi TRUOC moi cum ghi (43/38c/38d/41a da dinh).'''
    os.makedirs(RES39, exist_ok=True); os.makedirs(CKPT, exist_ok=True); return RES39
ensure_res()
json.dump({'run_tag': RUN_TAG, 'notebook': '39'}, open(os.path.join(RES39, '_run_info.json'), 'w'))
open(os.path.join(CKPT, '_keep'), 'w').write('keep')
LOCAL = '/content/cxrlt_518'

# --- CO CHINH ---
EHR_SRC     = 'col'        # 'col' = ColTokBag(217) | 'xgb' = ColTok(40 dac trung XGB tu 52a)
                           # CHON SAU KHI DOC nb53: neu `noi dung(xgb) - noi dung(col)` LON
                           # thi nut that la BO MA HOA -> dat 'xgb'.
ADAPT_AFTER = (8, 9)       # chen adapter SAU block 8 va 9 (1-based). `11` do I_location ~ 0
                           # giua (5,7) va (10,11) => vi tri trong dai giua khong quan trong.
                           # TRANH block 11-12: NOTE_07 goi do la "late fusion doi lot".
ARMS_ON     = ('D1', 'D2_U', 'D2_E', 'D3_U', 'D3_E')   # cat con 3 arm: bo 'D3_U','D3_E'
FILM_HID    = 256          # bottleneck — giu dang cua `11` (234,368 tham so) de so duoc

BB_RUN = 'a2_E3'
SEEDS  = (42,)
SEED, SIZE, EPOCHS = 42, 518, 10
BATCH, ACCUM = 32, 8       # 5 arm x graph block 9-12 -> nang. EFF 256. OOM thi 16x16.
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
W_FRONTAL, W_LATERAL = 0.7, 0.3
NW = min(8, (os.cpu_count() or 2))
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0
DO_TEST, N_TEST_EP = True, 3

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

BB_SRC = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
for p, hd in ((BB_SRC, '38a'), (P_MICRO, '29c'), (P_VOCAB, '29c'), (P_OOF, '52a')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
YM = Y[LAB40].to_numpy('float32')
P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
V_ = ((pd.read_parquet(P_VIEW)[['dicom_id','view_group','use']]
         .set_index('dicom_id').loc[ORDER.dicom_id].reset_index()) if os.path.exists(P_VIEW)
      else ORDER[['dicom_id','view_group']].assign(use=lambda d: d.view_group != 'unknown'))
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL       # CUNG cach cat nhu 35/37/38*/44*/45/52*/53
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,} | '
      f'test {len(I_TE):,}')
print(f'EHR_SRC={EHR_SRC} | adapter SAU block {ADAPT_AFTER} | {len(ARMS_ON)} arm | '
      f'E={EPOCHS} | BATCH {BATCH}x{ACCUM}')

Mounted at /content/drive
thiet bi: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102 GB | AMP torch.bfloat16
377,110 anh | khop 219,785 | chon 39,086 | val 39,293 | test 78,946
EHR_SRC=col | adapter SAU block (8, 9) | 5 arm | E=10 | BATCH 32x8


## C1 · EHR — verbatim `53`

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
_m = (E_DF.set_index('study_id')[[c for c in SPEC['Q_FULL'] if c in E_DF.columns]].reindex(_sid)
          .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))
_m = np.nan_to_num(_m, nan=0.0)
COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468 -> softmax bao hoa -> NaN
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
COLS = [COLS[i] for i in _keep]; EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
_VOC = json.load(open(P_VOCAB)); V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all()
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TF_].mean(0, keepdims=True)) / (_l[TF_].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
FEAT = {'col': EHR_COL, 'xgb': XGB_F}[EHR_SRC]
USE_BAG = EHR_SRC == 'col'      # 40 dac trung XGB da tom tat CA vi khuan -> khong them tui nua
print(f'EHR_SRC={EHR_SRC} | FEAT {FEAT.shape} | tui vi khuan {"CO" if USE_BAG else "KHONG"} '
      f'(V={V_MIC-1}, dai {MIC_L})')
# Hang TEST cua xgb_oof la HANG SO prevalence (52a co y khong du doan test) -> C9 loai arm xgb
# khoi bang test neu EHR_SRC='xgb'.

EHR_SRC=col | FEAT (377110, 217) | tui vi khuan CO (V=305, dai 10)


## C2 · Ảnh + backbone + **cổng cắt đôi forward**

Rủi ro cài đặt lớn nhất: cắt HF model làm đôi đòi biết nội tạng (`embeddings` → `encoder.layer[…]`
→ `layernorm`). Sai một bước là mọi số vô nghĩa **mà không báo lỗi**.

`cổng 1` chạy forward nguyên vẹn và forward cắt đôi trên cùng batch, assert bằng nhau. Không đạt
thì dừng ngay.

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95)
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()   # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

_ck = torch.load(BB_SRC, map_location='cpu', weights_only=False)
_sd0 = {k: v.clone() for k, v in AutoModel.from_pretrained('microsoft/rad-dino').state_dict().items()}
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
with torch.no_grad():
    _d = max((_sd0[k] - v).abs().max().item() for k, v in backbone.state_dict().items())
del _sd0
assert _d > 1e-6, f'backbone GIONG HET pretrain (lech {_d:.1e}) -> `ft` khong duoc nap'
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
LAYERS = backbone.encoder.layer
NL = len(LAYERS)
print(f'BACKBONE `ft` 38a/{BB_RUN} | {NL} block | d={D_CXR} | lech pretrain {_d:.3f}')
assert max(ADAPT_AFTER) < NL, f'ADAPT_AFTER {ADAPT_AFTER} vuot {NL} block'
GRAD_FROM = min(ADAPT_AFTER)     # block tu day tro len phai CO graph

def _out(r):
    '''Dinov2Layer tra ve Tensor TRAN o ban transformers nay, KHONG phai tuple.
       `r[0]` tren Tensor lay MAU DAU TIEN cua batch -> [B,N,D] thanh [N,D], broadcasting
       che lap loi nen no khong bao gi. Phai kiem isinstance.'''
    return r[0] if isinstance(r, tuple) else r

def run_blocks(h, lo, hi):
    for i in range(lo, hi): h = _out(LAYERS[i](h))
    return h


class DS(Dataset):
    def __init__(s, idx, tf, ehr_perm=None): s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k])
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return dict(image=s.tf(Image.merge('RGB', (im, im, im))), y=torch.from_numpy(YM[i]),
                    row=i, pos=k, erow=(i if s.perm is None else int(s.perm[k])))

def loader(idx, tf, shuffle, bs, perm=None, nw=NW, drop_last=None):
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle if drop_last is None else drop_last,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

# --- cong 1: forward CAT DOI phai bang forward NGUYEN VEN ---
_b0 = next(iter(loader(I_VA[:4], eval_tf, False, 4, nw=0)))
_x = _b0['image'].to(dev)
with torch.no_grad():
    _full = backbone(pixel_values=_x).last_hidden_state
    _h = backbone.embeddings(_x)
    _h = run_blocks(_h, 0, NL)
    _split = backbone.layernorm(_h)
_e1 = float((_full - _split).abs().max())
assert _e1 < 1e-3, (f'forward CAT DOI lech {_e1:.2e} so voi nguyen ven -> da cat SAI cho trong '
                    'kien truc HF. DUNG chay tiep, moi so sau day vo nghia.')
N_TOK = _full.shape[1] - 1
assert N_TOK == (SIZE // backbone.config.patch_size) ** 2
print(f'cong 1: forward cat doi khop nguyen ven toi {_e1:.1e} | {N_TOK} patch token')
print(f'  block 0..{GRAD_FROM-1} chay no_grad (CHUNG cho moi arm)')
print(f'  block {GRAD_FROM}..{NL-1} chay CO graph, RIENG tung arm co adapter')

anh local: 377,110 / 377,110


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

BACKBONE `ft` 38a/a2_E3 | 12 block | d=768 | lech pretrain 0.021
cong 1: forward cat doi khop nguyen ven toi 0.0e+00 | 1369 patch token
  block 0..7 chay no_grad (CHUNG cho moi arm)
  block 8..11 chay CO graph, RIENG tung arm co adapter


In [ ]:
# ============ TIM CHO CAT SAI ============
print(f'type: {type(backbone).__name__} | model_type: {backbone.config.model_type} | '
      f'{backbone.config.num_hidden_layers} block')
print('con truc tiep:', [n for n, _ in backbone.named_children()])
print('encoder:', [n for n, _ in backbone.encoder.named_children()])
print(f'so layer: {len(backbone.encoder.layer)}')
_l0 = backbone.encoder.layer[0]
print(f'layer[0]: {type(_l0).__name__} | con: {[n for n, _ in _l0.named_children()]}')

with torch.no_grad():
    o = backbone(pixel_values=_x, output_hidden_states=True)
    hs = o.hidden_states                 # (sau embeddings, sau block 1, ..., sau block N)
    print(f'\nhidden_states: {len(hs)} phan tu, moi cai {tuple(hs[0].shape)}')
    h = backbone.embeddings(_x)
    print(f'\n{"buoc":<22} {"lech so voi HF":>16}')
    print(f'{"sau embeddings":<22} {float((h - hs[0]).abs().max()):>16.3e}')
    for i in range(len(backbone.encoder.layer)):
        r = backbone.encoder.layer[i](h)
        h = r[0] if isinstance(r, tuple) else r
        print(f'{"sau block " + str(i):<22} {float((h - hs[i+1]).abs().max()):>16.3e}'
              + ('   <- BAT DAU LECH' if float((h - hs[i+1]).abs().max()) > 1e-4
                 and float((hs[i] - hs[i]).abs().max()) == 0 else ''))
    print(f'{"sau layernorm":<22} '
          f'{float((backbone.layernorm(h) - o.last_hidden_state).abs().max()):>16.3e}')
    print(f'\nkieu tra ve cua layer[0]: {type(backbone.encoder.layer[0](hs[0]))}')


type: Dinov2Model | model_type: dinov2 | 12 block
con truc tiep: ['embeddings', 'encoder', 'layernorm']
encoder: ['layer']
so layer: 12
layer[0]: Dinov2Layer | con: ['norm1', 'attention', 'layer_scale1', 'drop_path', 'norm2', 'mlp', 'layer_scale2']

hidden_states: 13 phan tu, moi cai (4, 1370, 768)

buoc                     lech so voi HF
sau embeddings                0.000e+00
sau block 0                   0.000e+00
sau block 1                   0.000e+00
sau block 2                   0.000e+00
sau block 3                   0.000e+00
sau block 4                   0.000e+00
sau block 5                   0.000e+00
sau block 6                   0.000e+00
sau block 7                   0.000e+00
sau block 8                   0.000e+00
sau block 9                   0.000e+00
sau block 10                  0.000e+00
sau block 11                  0.000e+00
sau layernorm                 0.000e+00

kieu tra ve cua layer[0]: <class 'torch.Tensor'>


## C3 · Kiến trúc — hai hình dạng adapter, **cùng vị trí**, cùng bộ mã hoá

Cả hai lấy đầu ra của **cùng** `ColTokBag`. Khác nhau đúng ở cách dùng:

```
FiLM   e = mean(token)  ->  gamma, beta in R^768  ->  Z*(1+gamma) + beta      TOAN CUC
XAttn  token nguyen ven ->  Z + pr(ln_a(XAttn(Q=Z, K/V=token)))               THEO KHONG GIAN
```

`gamma`/`beta` và `pr` đều **zero-init** ⇒ arm khởi động **đúng bằng** `D1`. Chỉ **một** hệ số
zero-init mỗi nhánh: `06b` v1 đặt `g=0` **và** `pe=0` tạo điểm yên ngựa chết
(`dL/dg = pe(he) = 0` và `dL/dpe ~ g = 0`).

In [ ]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    def __init__(s, V, d, const=False):
        super().__init__()
        s.const = const; s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w):
        if s.const:      # MOT chi so, khong phai MIC_L ban sao (tui that TB 1.7 vi khuan)
            ids = torch.zeros(ids.shape[0], 1, dtype=torch.long, device=ids.device)
            w = torch.ones(ids.shape[0], 1, device=ids.device)
        return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    '''d_in token COT + 1 token TONG HOP + 1 token BAG (tuy chon). `const=True` -> VECTOR HANG SO
       hoc duoc, khong doc benh nhan (mau H4z/H6z cua 06b).'''
    def __init__(s, d_in, d=256, const=False, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.const = const
        if const: s.c = nn.Parameter(torch.randn(d_in + 1, d) * 0.02)
        s.mb = MicroBag(V, d, const) if V else None
        if s.mb is not None:
            s.bln = nn.LayerNorm(d); s.cln = nn.LayerNorm(d) if const else nn.Identity()
    def forward(s, e, mids=None, mcnt=None):
        if s.const: base = s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        else:
            col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
            base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([s.cln(base), s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class FiLMAdapter(nn.Module):
    '''Z' = Z * (1 + gamma(e)) + beta(e), e = mean(token EHR). Dieu bien TOAN CUC: mot cap
       (gamma, beta) cho ca anh. `06` do FiLM ~ concat (p = 0.308) O DAU RA — nhung o day no nam
       TRONG backbone, con 4 block attention phia sau, nen cung phep toan lai la lop ham khac.
       Bottleneck giu dang cua `11` (234,368 tham so) de so duoc voi G_mid = +0.0036.'''
    def __init__(s, d, d_e, hid=256):
        super().__init__()
        s.mlp = nn.Sequential(nn.Linear(d_e, hid), nn.GELU(), nn.Linear(hid, 2 * d))
        nn.init.zeros_(s.mlp[-1].weight); nn.init.zeros_(s.mlp[-1].bias)   # zero-init -> Z' = Z
    def forward(s, z, etok):
        g, b = s.mlp(etok.mean(1)).chunk(2, -1)
        return z * (1 + g.unsqueeze(1)) + b.unsqueeze(1)
    def gb(s, etok):
        g, b = s.mlp(etok.mean(1)).chunk(2, -1); return g, b

class XAttnAdapter(nn.Module):
    '''Z' = Z + pr(ln_a(XAttn(Q = 1370 token anh, K/V = token EHR))). Cung DANG `Tier1` cua 53
       nhung dat TRONG backbone. softmax chi tren token EHR nen EHR khong the bi token anh nhan
       chim (co che that bai cua 06b H6). `ln_a` chuan hoa DAU VAO cua phep chieu — ban va 38d,
       thieu no thi arm doi chung sup 0.2416 -> 0.0681.'''
    def __init__(s, d, d_e, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.ein = nn.Linear(d_e, d)
        s.pr = nn.Linear(d, d); nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, z, etok):
        t = s.ein(etok)
        a, _ = s.at(s.n(z), t, t, need_weights=False)
        return z + s.pr(s.ln_a(a))

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = patch token.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop); s.out = GroupFC(C, d)
    def forward(s, patch, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        h, w = s.img(q, s.pin(patch), need_w); return s.out(h), w

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*/45/52*/53. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

kien truc san sang


## C4 · 5 arm + cổng khoá trước

In [ ]:
ARMS = {'D1':   dict(shape=None,   const=False),
        'D2_U': dict(shape='film', const=True),
        'D2_E': dict(shape='film', const=False),
        'D3_U': dict(shape='xattn', const=True),
        'D3_E': dict(shape='xattn', const=False)}
ARMS = {k: v for k, v in ARMS.items() if k in ARMS_ON}
MK = {'film': FiLMAdapter, 'xattn': XAttnAdapter}

def build_arms(seed=SEED):
    M, E, A = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)     # gieo lai TRUOC TUNG arm -> khoi tao giong het nhau
        M[tag] = MLDecoder(C, D_CXR, D_MLD)
        if cfg['shape'] is None: continue
        E[tag] = ColTokBag(FEAT.shape[1], D_ETOK, cfg['const'], V=(V_MIC if USE_BAG else None))
        kw = dict(hid=FILM_HID) if cfg['shape'] == 'film' else {}
        A[tag] = nn.ModuleList([MK[cfg['shape']](D_CXR, D_ETOK, **kw) for _ in ADAPT_AFTER])
    return M.to(dev), E.to(dev), A.to(dev)

MOD, ETK, ADP = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + ([ETK[t]] if t in ETK else [])
                                  + ([ADP[t]] if t in ADP else []))
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<7} {"hinh dang":<7} {"tham so":>12}  {"rieng adapter":>14}')
for t, c in ARMS.items():
    na = sum(p.numel() for p in ADP[t].parameters()) if t in ADP else 0
    print(f'  {t:<5} {str(c["shape"]):<7} {NP_[t]:>12,}  {na:>14,}')
print(f'  (`11` dung bottleneck 234,368 tham so cho adapter)')

_e = torch.randn(4, FEAT.shape[1], device=dev)
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev); _mc = torch.zeros(4, MIC_L, device=dev)
_z = torch.randn(4, N_TOK + 1, D_CXR, device=dev)
tok = lambda t: ETK[t](_e, _mi, _mc)

assert not any(p.requires_grad for p in backbone.parameters()), 'backbone chua dong cung'
assert not MOD[list(ARMS)[0]].q.weight.requires_grad, 'query_embed dang hoc'
# --- cong 2: MOI adapter zero-init => Z' = Z tuyet doi ---
with torch.no_grad():
    for t in ADP:
        for a in ADP[t]:
            _d2 = float((a(_z, tok(t)) - _z).abs().max())
            assert _d2 < 1e-6, f'{t}: adapter KHONG zero-init (lech {_d2:.2e})'
print(f'\ncong 2: moi adapter zero-init OK -> {len(ADP)} arm khoi dong DUNG BANG D1')
# --- cong 3: khop tham so tung cap that/doi-chung ---
for a, b in (('D2_E','D2_U'), ('D3_E','D3_U')):
    if a in ARMS and b in ARMS:
        assert NP_[b] >= NP_[a], f'{b} IT tham so hon {a} -> doi chung khong con bao thu'
        print(f'cong 3: {b} {NP_[b]:,} >= {a} {NP_[a]:,} (+{NP_[b]-NP_[a]:,}) -> `{a}-{b}>0` BAO THU')
# --- cong 4: hai hinh dang phai SINH RA dieu bien KHAC nhau (bat loi dispatch) ---
if 'D2_E' in ARMS and 'D3_E' in ARMS:
    with torch.no_grad():
        for a in list(ADP['D2_E']) + list(ADP['D3_E']):
            for p in a.parameters(): p.data.normal_(0, 0.02)
        _o2 = ADP['D2_E'][0](_z, tok('D2_E')); _o3 = ADP['D3_E'][0](_z, tok('D3_E'))
        _dd = float((_o2 - _o3).abs().max())
    assert _dd > 1e-4, f'FiLM va XAttn cho ra dieu bien GIONG HET (lech {_dd:.1e}) -> re nhanh sai'
    print(f'cong 4: FiLM va XAttn sinh dieu bien KHAC nhau OK (lech {_dd:.2f})')
    MOD, ETK, ADP = build_arms()      # dung lai sau khi da pha trong so de kiem
print(f'\nadapter dat SAU block {ADAPT_AFTER} | block {GRAD_FROM}..{NL-1} co graph')

arm     hinh dang      tham so   rieng adapter
  D1    None       2,517,032               0
  D2_U  film       3,872,296         921,088
  D2_E  film       3,815,976         921,088
  D3_U  xattn      9,258,024       6,306,816
  D3_E  xattn      9,201,704       6,306,816
  (`11` dung bottleneck 234,368 tham so cho adapter)

cong 2: moi adapter zero-init OK -> 4 arm khoi dong DUNG BANG D1
cong 3: D2_U 3,872,296 >= D2_E 3,815,976 (+56,320) -> `D2_E-D2_U>0` BAO THU
cong 3: D3_U 9,258,024 >= D3_E 9,201,704 (+56,320) -> `D3_E-D3_U>0` BAO THU
cong 4: FiLM va XAttn sinh dieu bien KHAC nhau OK (lech 0.31)

adapter dat SAU block (8, 9) | block 8..11 co graph


## C5 · Chấm điểm — verbatim `53`

In [ ]:
from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}
_rs6 = np.random.default_rng(6); _yy = (_rs6.random((3000, C)) < 0.2).astype('float32')
_ss6 = _rs6.random((3000, C)).astype('float32')
_dd6 = float(np.nanmax(np.abs(ap_cols(_yy, _ss6) -
        np.array([average_precision_score(_yy[:, j], _ss6[:, j]) for j in range(C)]))))
assert _dd6 < 1e-6, f'ap_cols lech sklearn {_dd6:.2e}'
print(f'cong 5: ap_cols khop sklearn toi {_dd6:.2e}')

cong 5: ap_cols khop sklearn toi 1.69e-08


## C6 · Huấn luyện — block `0..7` **dùng chung**, block `8..11` **riêng từng arm**

Đây là notebook **duy nhất** mà các arm không dùng chung toàn bộ forward. Adapter đổi `Z_8` nên
block phía trên phải chạy lại cho từng arm, và phải **có graph** để gradient về được adapter (tham
số backbone vẫn đông cứng).

`D1` gần như miễn phí: không adapter ⇒ block trên chạy `no_grad`, chỉ đầu đọc có gradient.

In [9]:
crit = ASL(gn=4, gp=0).to(dev)
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = torch.from_numpy(FEAT)

def arm_forward(t, h_lo, erow):
    '''h_lo = Z sau block GRAD_FROM-1 (da tinh CHUNG, no_grad). Tra ve logit cua arm `t`.'''
    cfg = ARMS[t]
    if cfg['shape'] is None:
        with torch.no_grad():
            h = backbone.layernorm(run_blocks(h_lo, GRAD_FROM, NL))
        return MOD[t](h[:, 1:])[0]
    e = FEAT_T[erow].to(dev)
    et = ETK[t](e, MIDS_T[erow].to(dev), MCNT_T[erow].to(dev))
    h = h_lo
    ai = 0
    for i in range(GRAD_FROM, NL):
        if i in ADAPT_AFTER: h = ADP[t][ai](h, et); ai += 1
        h = _out(LAYERS[i](h))
    return MOD[t](backbone.layernorm(h)[:, 1:])[0]


@torch.no_grad()
def shared_lo(img):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        h = backbone.embeddings(img)
        return run_blocks(h, 0, GRAD_FROM).float()

@torch.no_grad()
def predict(idx, perm=None, bs=32):
    MOD.eval(); ETK.eval(); ADP.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    for b in tqdm(loader(idx, eval_tf, False, bs, perm, nw=0, drop_last=False),
                  desc='eval', leave=False):
        hl = shared_lo(b['image'].to(dev, non_blocking=True))
        for t in ARMS:
            P[t][b['pos'].numpy()] = torch.sigmoid(
                arm_forward(t, hl, b['erow']).float()).cpu().numpy()
    assert not any(np.isnan(v).any() for v in P.values()), 'predict cho ra NaN'
    for t in ARMS: assert np.abs(P[t]).sum(1).min() > 0, f'{t}: hang khong duoc ghi'
    return P

P_LOG = os.path.join(RES39, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
DL = loader(I_TF, train_tf, True, BATCH)

for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'): print(f'{sk}: da xong, bo qua'); continue
    MOD, ETK, ADP = build_arms(seed)
    PARAM_OF = {t: [p for m in ([MOD[t]] + ([ETK[t]] if t in ETK else [])
                                + ([ADP[t]] if t in ADP else []))
                    for p in m.parameters() if p.requires_grad] for t in ARMS}
    N_CLIP = {t: 0 for t in ARMS}; N_SKIP = {t: 0 for t in ARMS}; N_STEP = 0
    rest = [p for m in (MOD, ETK, ADP) for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{'params': [p for p in rest if p.ndim > 1], 'weight_decay': WD},
                             {'params': [p for p in rest if p.ndim <= 1], 'weight_decay': 0.0}], lr=LR)
    steps = math.ceil(len(I_TF) / (BATCH * ACCUM)) * EPOCHS; warm = int(WARMUP * steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    LOG.setdefault(sk, {})
    P_RES = os.path.join(CKPT, f'{sk}_resume.pt'); ep0 = 0
    if os.path.exists(P_RES):
        _r = torch.load(P_RES, map_location=dev, weights_only=False)
        _old = _r.get('arms')
        assert _old is None or list(_old) == list(ARMS), (
            f'checkpoint co {len(_old)} arm {list(_old)}, notebook co {len(ARMS)} arm.\n'
            '  -> DOI RUN_TAG o C0 (vd "r2") roi chay lai tu C0. KHONG xoa gi.')
        MOD.load_state_dict(_r['mod']); ETK.load_state_dict(_r['etk']); ADP.load_state_dict(_r['adp'])
        opt.load_state_dict(_r['opt']); sch.load_state_dict(_r['sch'])
        N_CLIP, N_SKIP, N_STEP, ep0 = _r['n_clip'], _r['n_skip'], _r['n_step'], _r['ep'] + 1
        print(f'{sk}: resume tu epoch {ep0+1}/{EPOCHS}')

    for ep in range(ep0, EPOCHS):
        MOD.train(); ETK.train(); ADP.train()
        t0, tot, nb = time.time(), {t: 0.0 for t in ARMS}, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(DL, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            hl = shared_lo(b['image'].to(dev, non_blocking=True)); y = b['y'].to(dev)
            loss, bad = 0.0, []
            for t in ARMS:
                lg = arm_forward(t, hl, b['erow'])
                if not torch.isfinite(lg).all(): bad.append(t); N_SKIP[t] += 1; continue
                l_ = crit(lg.float(), y); loss = loss + l_; tot[t] += float(l_.detach())
            if bad and (sum(N_SKIP.values()) <= 3 or j % 400 == 0):
                print(f'   [{sk} ep{ep+1} buoc {j}] logit khong huu han: {bad}')
            if not torch.is_tensor(loss): opt.zero_grad(set_to_none=True); continue
            (loss / ACCUM).backward(); nb += 1
            if (j + 1) % ACCUM == 0:
                # Kiem gradient TRUOC khi buoc: total_norm = nan cho clip_coef = nan va BOI BAN
                # toan bo tham so -> mot buoc hong lam chet vinh vien.
                for t, ps in PARAM_OF.items():
                    gn = torch.nn.utils.clip_grad_norm_(ps, CLIP, error_if_nonfinite=False)
                    if not torch.isfinite(gn):
                        for p in ps:
                            if p.grad is not None: p.grad.zero_()
                        N_SKIP[t] += 1
                    elif float(gn) > CLIP: N_CLIP[t] += 1
                N_STEP += 1; opt.step(); sch.step(); opt.zero_grad(set_to_none=True)

        PV = predict(I_VA); PS = predict(I_SL); ensure_res()
        for t, p in PV.items(): np.save(os.path.join(RES39, f'p_{t}_{sk}_ep{ep+1}.npy'), p)
        for t, p in PS.items(): np.save(os.path.join(RES39, f'psl_{t}_{sk}_ep{ep+1}.npy'), p)
        if DO_TEST and ep >= EPOCHS - N_TEST_EP:
            PT = predict(I_TE); ensure_res()
            for t, p in PT.items(): np.save(os.path.join(RES39, f'pte_{t}_{sk}_ep{ep+1}.npy'), p)
            with open(P_TOUCH, 'a') as _f:
                _f.write(json.dumps(dict(nb='39', run=RUN_TAG, seed=seed, ep=ep+1, n=len(I_TE),
                                         arms=list(ARMS), muc_dich='theo doi',
                                         t=datetime.datetime.now().isoformat(timespec='seconds')))+'\n')
            print(f'   [TEST] infer {len(I_TE):,} anh -> pte_*_ep{ep+1}.npy + _test_touches.jsonl')
        _e2 = {}
        for t, p in PV.items():
            ap = ap_cols(YM[VA], p)
            _e2[t] = dict(mAP_anh=float(np.nanmean(ap)), **tiers(ap),
                          sel=float(np.nanmean(ap_cols(YM[SL], PS[t]))))
        LOG[sk][str(ep)] = _e2; json.dump(LOG, open(P_LOG, 'w'))
        torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), adp=ADP.state_dict(),
                        opt=opt.state_dict(), sch=sch.state_dict(), n_clip=N_CLIP, n_skip=N_SKIP,
                        n_step=N_STEP, ep=ep, seed=seed, arms=list(ARMS)), P_RES)
        g = lambda a, b: _e2[a]['mAP_anh'] - _e2[b]['mAP_anh']
        print(f'{sk} ep{ep+1:>2} | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | '
              f'{(time.time()-t0)/60:.1f} ph | loss ' +
              ' '.join(f'{t}:{tot[t]/max(nb,1):.3f}' for t in ARMS))
        print('   VAL      ' + ' '.join(f'{t:>6}:{_e2[t]["mAP_anh"]:.4f}' for t in ARMS))
        print('   LAT CHON ' + ' '.join(f'{t:>6}:{_e2[t]["sel"]:.4f}' for t in ARMS))
        print('   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`')
        if 'D2_E' in ARMS: print(f'   G_film  = D2_E - D2_U = {g("D2_E","D2_U"):+.4f}   '
                                 f'(11 do G_mid +0.0036 tren cohort CU 13 nhan)')
        if 'D3_E' in ARMS: print(f'   G_xattn = D3_E - D3_U = {g("D3_E","D3_U"):+.4f}')
        if 'D2_E' in ARMS and 'D3_E' in ARMS:
            print(f'   G_xattn - G_film      = {g("D3_E","D3_U")-g("D2_E","D2_U"):+.4f}  '
                  '<- HINH DANG co quan trong khong')
        if 'D2_U' in ARMS: print(f'   capacity = D2_U - D1  = {g("D2_U","D1"):+.4f}  '
                                 '(phan CHI do them tham so)')
        _f = {t: N_CLIP[t]/max(N_STEP,1) for t in ARMS}
        if max(_f.values()) > 0.01:
            print('   clip: ' + ' '.join(f'{t}:{v:.0%}' for t, v in _f.items() if v > 0.01))
        if max(N_SKIP.values()) > 0:
            print('   BUOC BI BO: ' + ' '.join(f'{t}:{v}' for t, v in N_SKIP.items() if v > 0))
    ensure_res()
    torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), adp=ADP.state_dict(),
                    seed=seed, arms=list(ARMS)), os.path.join(CKPT, f'{sk}.pt'))
    LOG[sk]['done'] = True; json.dump(LOG, open(P_LOG, 'w'))
    print(f'{sk}: xong')
print('tat ca seed xong')

s42 ep1/10:   0%|          | 0/6868 [00:00<?, ?it/s]

eval:   0%|          | 0/1228 [00:00<?, ?it/s]

eval:   0%|          | 0/1222 [00:00<?, ?it/s]

s42 ep 1 | 29 img/s | 127.4 ph | loss D1:0.043 D2_U:0.043 D2_E:0.043 D3_U:0.043 D3_E:0.043
   VAL          D1:0.2370   D2_U:0.2359   D2_E:0.2374   D3_U:0.2352   D3_E:0.2384
   LAT CHON     D1:0.2308   D2_U:0.2313   D2_E:0.2326   D3_U:0.2314   D3_E:0.2324
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
   G_film  = D2_E - D2_U = +0.0015   (11 do G_mid +0.0036 tren cohort CU 13 nhan)
   G_xattn = D3_E - D3_U = +0.0032
   G_xattn - G_film      = +0.0017  <- HINH DANG co quan trong khong
   capacity = D2_U - D1  = -0.0010  (phan CHI do them tham so)


s42 ep2/10:   0%|          | 0/6868 [00:00<?, ?it/s]

...

s42 ep10/10:   0%|          | 0/6868 [00:00<?, ?it/s]

eval:   0%|          | 0/1228 [00:00<?, ?it/s]

eval:   0%|          | 0/1222 [00:00<?, ?it/s]

eval:   0%|          | 0/2468 [00:00<?, ?it/s]

   [TEST] infer 78,946 anh -> pte_*_ep10.npy + _test_touches.jsonl
s42 ep10 | 25 img/s | 146.6 ph | loss D1:0.036 D2_U:0.036 D2_E:0.035 D3_U:0.036 D3_E:0.036
   VAL          D1:0.2392   D2_U:0.2382   D2_E:0.2402   D3_U:0.2396   D3_E:0.2413
   LAT CHON     D1:0.2342   D2_U:0.2333   D2_E:0.2351   D3_U:0.2336   D3_E:0.2359
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
   G_film  = D2_E - D2_U = +0.0020   (11 do G_mid +0.0036 tren cohort CU 13 nhan)
   G_xattn = D3_E - D3_U = +0.0017
   G_xattn - G_film      = -0.0003  <- HINH DANG co quan trong khong
   capacity = D2_U - D1  = -0.0010  (phan CHI do them tham so)
s42: xong
tat ca seed xong


## C7 · Kết quả — cổng khoá trước đặt trên `G_*`, không phải `arm − D1`

In [10]:
LOG = json.load(open(P_LOG)); EPk = str(EPOCHS - 1)
R = pd.DataFrame([dict(seed=s, arm=t, **v) for s in SEEDS if EPk in LOG.get(f's{s}', {})
                  for t, v in LOG[f's{s}'][EPk].items()])
assert len(R), 'chua co ket qua nao'
SD = [s for s in SEEDS if EPk in LOG.get(f's{s}', {})]
piv = R.pivot_table(index='arm', values=['mAP_anh','head','medium','tail','sel'],
                    aggfunc='mean').reindex(list(ARMS))
piv['tham_so'] = [NP_[t] for t in ARMS]
piv['bien_do'] = R.groupby('arm').mAP_anh.agg(lambda s: s.max()-s.min()).reindex(list(ARMS))
print(f'=== TUYET DOI (TB qua {len(SD)} seed) ===')
print(piv.to_string(float_format=lambda v: f'{v:.4f}'))

g = lambda a, s: float(R[(R.arm == a) & (R.seed == s)].mAP_anh.iloc[0])
def d(a, b):
    v = [g(a, s) - g(b, s) for s in SD]
    bd = (max(v) - min(v)) if len(v) > 1 else float('nan')
    return float(np.mean(v)), bd
G_FILM = d('D2_E', 'D2_U') if 'D2_E' in ARMS else (float('nan'),) * 2
G_XATT = d('D3_E', 'D3_U') if 'D3_E' in ARMS else (float('nan'),) * 2
CAP_F  = d('D2_U', 'D1') if 'D2_U' in ARMS else (float('nan'),) * 2
CAP_X  = d('D3_U', 'D1') if 'D3_U' in ARMS else (float('nan'),) * 2
print('\n=== NOI DUNG THUAN (arm - doi chung KHOP THAM SO) ===')
print(f'  G_film  = D2_E - D2_U  {G_FILM[0]:+.4f}  (bien do {G_FILM[1]:.4f})')
print(f'  G_xattn = D3_E - D3_U  {G_XATT[0]:+.4f}  (bien do {G_XATT[1]:.4f})')
print(f'\n  HINH DANG: G_xattn - G_film = {G_XATT[0]-G_FILM[0]:+.4f}')
print('\n=== DUNG LUONG (chi do them tham so, KHONG lien quan EHR) ===')
print(f'  D2_U - D1  {CAP_F[0]:+.4f}   |   D3_U - D1  {CAP_X[0]:+.4f}')
print('\n' + '=' * 70)
print('CONG KHOA TRUOC, dat tren G_*, KHONG phai `arm - D1`:')
print('  G < +0.002          -> DONG huong EHR fusion cho 40 nhan CXR-LT')
print('    +0.002 .. +0.004  -> vung xam, can seed thu hai + CI')
print('  G > +0.004          -> mo tiep')
for nm, (gv, _) in (('film', G_FILM), ('xattn', G_XATT)):
    if np.isnan(gv): continue
    v = 'DONG' if gv < 0.002 else ('vung xam' if gv < 0.004 else 'MO TIEP')
    print(f'  G_{nm:<6} {gv:+.4f}  ->  {v}')
print('=' * 70)
print('\nDOC: neu `capacity` LON ma `G` ~ 0 thi ket luan la "them adapter co ich, nhung EHR khong"')
print('     — phat bieu HOAN TOAN KHAC voi "deep conditioning co tac dung". Chi cap arm nay')
print('     moi phan biet duoc (06b: H6z 0.4330 vs H6 0.4339, trong khi H5 0.4336).')
print(f'\nmoc doi chieu: `11` G_mid +0.0036 (cohort CU, 13 nhan, DUNG MOT nhan tail, 2/3 seed)')
print(f'               `52b` t1_col - t1_col_zero = -0.0015 (tang NGOAI, 40 nhan)')
ensure_res(); R.to_csv(os.path.join(RES39, 'main_table.csv'), index=False)

=== TUYET DOI (TB qua 1 seed) ===
       head  mAP_anh  medium    sel   tail  tham_so  bien_do
arm                                                         
D1   0.4534   0.2392  0.1668 0.2342 0.0912  2517032   0.0000
D2_U 0.4529   0.2382  0.1673 0.2333 0.0876  3872296   0.0000
D2_E 0.4582   0.2402  0.1714 0.2351 0.0826  3815976   0.0000
D3_U 0.4526   0.2396  0.1659 0.2336 0.0952  9258024   0.0000
D3_E 0.4571   0.2413  0.1697 0.2359 0.0905  9201704   0.0000

=== NOI DUNG THUAN (arm - doi chung KHOP THAM SO) ===
  G_film  = D2_E - D2_U  +0.0020  (bien do nan)
  G_xattn = D3_E - D3_U  +0.0017  (bien do nan)

  HINH DANG: G_xattn - G_film = -0.0003

=== DUNG LUONG (chi do them tham so, KHONG lien quan EHR) ===
  D2_U - D1  -0.0010   |   D3_U - D1  +0.0004

CONG KHOA TRUOC, dat tren G_*, KHONG phai `arm - D1`:
  G < +0.002          -> DONG huong EHR fusion cho 40 nhan CXR-LT
    +0.002 .. +0.004  -> vung xam, can seed thu hai + CI
  G > +0.004          -> mo tiep
  G_film   +0.0020  ->  vun

## C8 · Đo cơ chế — FiLM không có attention nên chẩn đoán khác

```
‖gamma(e)‖, ‖beta(e)‖     bien do dieu bien
sd giua CAC BENH NHAN     so voi arm `_U` (hang so) lam SAN — doc TY LE, khong doc so tuyet doi
ratio = ‖Z'-Z‖ / ‖Z‖      adapter dong gop bao nhieu vao bieu dien
```

`_U` dùng vector hằng số nên `sd/BN` của nó là **sàn**: `~1.0×` nghĩa là EHR **không tạo thêm biến
thiên nào**. Đây đúng cách `52b C13` đã dùng để chứng minh `Tier1` không phải mean trá hình.

In [11]:
NDIAG, BSD = 256, 16
_id = I_VA[np.random.default_rng(5).choice(len(I_VA), min(NDIAG, len(I_VA)), replace=False)]
ck = torch.load(os.path.join(CKPT, f's{SD[0]}.pt'), map_location=dev, weights_only=False)
MOD.load_state_dict(ck['mod']); ETK.load_state_dict(ck['etk']); ADP.load_state_dict(ck['adp'])
MOD.eval(); ETK.eval(); ADP.eval()
DIAG = {}
with torch.no_grad():
    for t in [x for x in ARMS if x in ADP]:
        GS, BS, RT = [], [], []
        for i in range(0, len(_id), BSD):
            b = next(iter(loader(_id[i:i+BSD], eval_tf, False, BSD, nw=0, drop_last=False)))
            hl = shared_lo(b['image'].to(dev))
            e = FEAT_T[b['erow']].to(dev)
            et = ETK[t](e, MIDS_T[b['erow']].to(dev), MCNT_T[b['erow']].to(dev))
            a0 = ADP[t][0]
            z1 = a0(hl, et)
            RT.append(float(((z1 - hl).norm(dim=-1) / (hl.norm(dim=-1) + 1e-9)).mean()))
            if isinstance(a0, FiLMAdapter):
                gm, bt = a0.gb(et); GS.append(gm.cpu()); BS.append(bt.cpu())
            else:
                GS.append((z1 - hl).mean(1).cpu())
        DIAG[t] = dict(ratio=float(np.mean(RT)),
                       sd_bn=float(torch.cat(GS).std(0).mean()),
                       mag=float(torch.cat(GS).abs().mean()),
                       mag_b=float(torch.cat(BS).abs().mean()) if BS else float('nan'))
print(f'{"arm":<7} {"ratio ||dZ||/||Z||":>19} {"|dieu bien| TB":>16} {"sd giua BN":>12}')
for t, v in DIAG.items():
    print(f'{t:<7} {v["ratio"]:>19.5f} {v["mag"]:>16.5f} {v["sd_bn"]:>12.5f}')
print('\n  doi chung MIEN PHI: arm `_U` dung HANG SO -> sd/BN cua no la SAN.')
for a, b in (('D2_E','D2_U'), ('D3_E','D3_U')):
    if a in DIAG and b in DIAG:
        print(f'  {a} / {b}:  sd/BN {DIAG[a]["sd_bn"]/max(DIAG[b]["sd_bn"],1e-12):>6.2f}x  |  '
              f'ratio {DIAG[a]["ratio"]/max(DIAG[b]["ratio"],1e-12):>5.2f}x')
print('  ~1.0x  =>  EHR KHONG tao them bien thien nao; moi thu la dung luong.')
print('  ratio ~ 0  =>  adapter khong dong gop gi vao bieu dien (zero-init chua thoat).')

arm      ratio ||dZ||/||Z||   |dieu bien| TB   sd giua BN
D2_U                0.16388          0.14032      0.00000
D2_E                0.24043          0.21612      0.17218
D3_U                0.05718          0.17113      0.00845
D3_E                0.11005          0.24245      0.06597

  doi chung MIEN PHI: arm `_U` dung HANG SO -> sd/BN cua no la SAN.
  D2_E / D2_U:  sd/BN 172179460525.51x  |  ratio  1.47x
  D3_E / D3_U:  sd/BN   7.80x  |  ratio  1.92x
  ~1.0x  =>  EHR KHONG tao them bien thien nao; moi thu la dung luong.
  ratio ~ 0  =>  adapter khong dong gop gi vao bieu dien (zero-init chua thoat).


## C9 · Tráo nhân quả + nhóm nhãn + TEST (chỉ theo dõi)

In [12]:
perm = np.random.default_rng(0).permutation(len(I_VA))
P0 = predict(I_VA); P1 = predict(I_VA, perm=I_VA[perm])
lgt = lambda p: np.log(p.clip(1e-6, 1-1e-6) / (1 - p.clip(1e-6, 1-1e-6)))
print(f'=== TRAO EHR SANG BENH NHAN KHAC (s{SD[0]}) — NHAN giu nguyen ===')
print('  Doc HAI cot: corr mot minh QUA CHAT (44b tung co corr = 1.000000 ma mat 0.0056 mAP).')
for t in ARMS:
    if ARMS[t]['shape'] is None: continue
    r = float(np.corrcoef(lgt(P0[t]).ravel(), lgt(P1[t]).ravel())[0, 1])
    dd = float(np.nanmean(ap_cols(YM[VA], P1[t])) - np.nanmean(ap_cols(YM[VA], P0[t])))
    v = ('EHR co tac dong that' if dd <= -0.002 else
         'EHR KHONG duoc dung' if r > 0.9999 and dd > -0.0005 else 'vung xam')
    print(f'  {t:<7} corr {r:.6f} | mAP khi trao {dd:+.4f}   {v}')
    if ARMS[t]['const']:
        assert r > 1 - 1e-9 and abs(dd) < 1e-9, f'{t} la doi chung HANG SO ma trao van doi -> ro ri'
print('  moc 06b: corr = 0.999934 (H6) | cong: doi chung hang so bat bien duoi phep trao')

INF = [l for l in LAB40 if re.search(r'pneumon|tubercul|infect|consolidat|abscess|empyema', l, re.I)]
I_INF = np.array([LAB40.index(l) for l in INF])
sk = f's{SD[0]}'; _r = []
for t in ARMS:
    f = os.path.join(RES39, f'p_{t}_{sk}_ep{EPOCHS}.npy')
    if not os.path.exists(f): continue
    a = ap_cols(YM[VA], np.load(f))
    _r.append(dict(arm=t, macro=float(np.nanmean(a)), nhiem_trung=float(np.nanmean(a[I_INF])),
                   tail=float(np.nanmean(a[TIER['tail']]))))
T = pd.DataFrame(_r).set_index('arm')
print(f'\nnhom nhiem trung ({len(INF)}): {INF}')
print(T.to_string(float_format=lambda v: f'{v:.4f}'))

if DO_TEST:
    _nlab = int((YM[TE].sum(0) > 0).sum())
    _rat = float(np.nanmean(YM[TE].mean(0) / (YM[VA].mean(0) + 1e-9)))
    print(f'\n=== TEST (CHI theo doi, KHONG tham gia lua chon nao) ===')
    print(f'cong: {_nlab}/{C} nhan co ca duong | prevalence(test)/prevalence(val) = {_rat:.3f}')
    if _nlab < 35 or not (0.5 <= _rat <= 2.0):
        print('  !! nhan test co ve BI GIU LAI hoac lech -> KHONG doc so test.')
    else:
        _sk = [t for t in ARMS if EHR_SRC == 'xgb' and ARMS[t]['shape'] is not None]
        if _sk: print(f'  LOAI (dac trung XGB tren test la HANG SO): {_sk}')
        rows = []
        for s in SD:
            for t in ARMS:
                if t in _sk: continue
                fs = [os.path.join(RES39, f'pte_{t}_s{s}_ep{e}.npy')
                      for e in range(EPOCHS - N_TEST_EP + 1, EPOCHS + 1)]
                fs = [f for f in fs if os.path.exists(f)]
                if not fs: continue
                a = ap_cols(YM[TE], np.mean([np.load(f) for f in fs], 0))   # luat last-3 cua 48
                rows.append(dict(seed=s, arm=t, test_last3=float(np.nanmean(a)),
                                 test_tail=float(np.nanmean(a[TIER['tail']]))))
        if rows:
            TT = pd.DataFrame(rows).pivot_table(index='arm', values=['test_last3','test_tail'],
                                                aggfunc='mean')
            TT['val'] = [piv.loc[a, 'mAP_anh'] for a in TT.index]
            print(TT.to_string(float_format=lambda v: f'{v:.4f}'))
            ensure_res(); TT.to_csv(os.path.join(RES39, 'test_table.csv'))
    print(f'\n  !! model chi thay {int(TF_.sum()):,} / {int((TR|VA).sum()):,} anh co nhan (THIEU 26%).')
    print('     So test o day la CAN DUOI, KHONG phai con so cuoi cho bai (do thuoc NHANH B).')

eval:   0%|          | 0/1228 [00:00<?, ?it/s]

eval:   0%|          | 0/1228 [00:00<?, ?it/s]

=== TRAO EHR SANG BENH NHAN KHAC (s42) — NHAN giu nguyen ===
  Doc HAI cot: corr mot minh QUA CHAT (44b tung co corr = 1.000000 ma mat 0.0056 mAP).
  D2_U    corr 1.000000 | mAP khi trao +0.0000   EHR KHONG duoc dung
  D2_E    corr 0.965911 | mAP khi trao -0.0138   EHR co tac dong that
  D3_U    corr 1.000000 | mAP khi trao +0.0000   EHR KHONG duoc dung
  D3_E    corr 0.992299 | mAP khi trao -0.0054   EHR co tac dong that
  moc 06b: corr = 0.999934 (H6) | cong: doi chung hang so bat bien duoi phep trao

nhom nhiem trung (3): ['Consolidation', 'Pneumonia', 'Tuberculosis']
      macro  nhiem_trung   tail
arm                            
D1   0.2392       0.1964 0.0912
D2_U 0.2382       0.1980 0.0876
D2_E 0.2402       0.2078 0.0826
D3_U 0.2396       0.1983 0.0952
D3_E 0.2413       0.2011 0.0905

=== TEST (CHI theo doi, KHONG tham gia lua chon nao) ===
cong: 40/40 nhan co ca duong | prevalence(test)/prevalence(val) = 1.119
      test_last3  test_tail    val
arm                              

## C10 · Ghi `REPORT_STAGE4_NB39.md`

In [13]:
L = []; w = L.append
w('# REPORT · `nb39` — deep conditioning: FiLM vs cross-attention, **cùng vị trí**'); w('')
w(f'`{RUN_TAG}` · backbone `ft` = `38a/{BB_RUN}` **đông cứng** · adapter sau block {ADAPT_AFTER} · '
  f'`EHR_SRC={EHR_SRC}` · `asl40` · `E={EPOCHS}` · **epoch cuối** · seeds {list(SD)} · '
  f'val {int(VA.sum()):,} ảnh.'); w('')
w('| arm | hình dạng | tham số | mAP ảnh | biên độ | tail | nhiễm trùng |')
w('|---|---|---|---|---|---|---|')
for t in ARMS:
    if t not in piv.index: continue
    r = piv.loc[t]
    w(f'| `{t}` | {ARMS[t]["shape"] or "—"} | {NP_[t]:,} | {r["mAP_anh"]:.4f} | '
      f'{r["bien_do"]:.4f} | {r["tail"]:.4f} | '
      + (f'{T.loc[t, "nhiem_trung"]:.4f}' if t in T.index else '—') + ' |')
w(''); w('```')
w(f'G_film  = D2_E - D2_U   {G_FILM[0]:+.4f}')
w(f'G_xattn = D3_E - D3_U   {G_XATT[0]:+.4f}')
w(f'HINH DANG: G_xattn - G_film   {G_XATT[0]-G_FILM[0]:+.4f}')
w(f'capacity: D2_U - D1 {CAP_F[0]:+.4f} | D3_U - D1 {CAP_X[0]:+.4f}')
w('```'); w('')
w('**Chỉ `G_*` được đọc.** `06b` chứng minh cặp arm này cứu kết luận khỏi sai (`H6z 0.4330` vs '
  '`H6 0.4339`, trong khi `H5 0.4336` khiến nó trông như có tác dụng); `52b` lặp lại đúng bài học '
  '(`t1_col_zero` **thắng** `t1_col` `+0.0015`).'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w(f'- **Một vị trí duy nhất** (sau block {ADAPT_AFTER}). `11` đo `I_location ≈ 0` giữa `(5,7)` và '
  '`(10,11)` nên vị trí trong dải giữa không quan trọng — nhưng đó là cohort **cũ**, 13 nhãn.')
w(f'- **Một bộ mã hoá** (`{EHR_SRC}`). `52b` cho thấy `ColTokBag` không mang nội dung ở tầng ngoài; '
  'nếu `nb53` xác nhận nút thắt là bộ mã hoá thì phải chạy lại với `EHR_SRC="xgb"`.')
w('- **`p_uni` chưa chạm** — tỉ lệ mẫu uni-modal là thay đổi lấy mẫu của cả vòng lặp, không làm '
  'được thành một arm. MedFuse đo `+0.015` từ đúng đòn bẩy này.')
w('- **Đích.** Nhãn CXR-LT do NLP trích từ **chính report của phim đó** ⇒ ảnh gần như đủ theo định '
  'nghĩa (`34`: nhãn tự động gọi thừa `Pneumonia` **5×**). Không con số nào ở đây nói gì về tiên '
  'lượng, tử vong, chuyển ICU, tái nhập viện.')
TXT = '\n'.join(L); ensure_res()
open(os.path.join(RES39, 'REPORT_STAGE4_NB39.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)

# REPORT · `nb39` — deep conditioning: FiLM vs cross-attention, **cùng vị trí**

`r1` · backbone `ft` = `38a/a2_E3` **đông cứng** · adapter sau block (8, 9) · `EHR_SRC=col` · `asl40` · `E=10` · **epoch cuối** · seeds [42] · val 39,293 ảnh.

| arm | hình dạng | tham số | mAP ảnh | biên độ | tail | nhiễm trùng |
|---|---|---|---|---|---|---|
| `D1` | — | 2,517,032 | 0.2392 | 0.0000 | 0.0912 | 0.1964 |
| `D2_U` | film | 3,872,296 | 0.2382 | 0.0000 | 0.0876 | 0.1980 |
| `D2_E` | film | 3,815,976 | 0.2402 | 0.0000 | 0.0826 | 0.2078 |
| `D3_U` | xattn | 9,258,024 | 0.2396 | 0.0000 | 0.0952 | 0.1983 |
| `D3_E` | xattn | 9,201,704 | 0.2413 | 0.0000 | 0.0905 | 0.2011 |

```
G_film  = D2_E - D2_U   +0.0020
G_xattn = D3_E - D3_U   +0.0017
HINH DANG: G_xattn - G_film   -0.0003
capacity: D2_U - D1 -0.0010 | D3_U - D1 +0.0004
```

**Chỉ `G_*` được đọc.** `06b` chứng minh cặp arm này cứu kết luận khỏi sai (`H6z 0.4330` vs `H6 0.4339`, trong khi `H5 0.4336` khiến nó trông như có tác dụng); `52b` lặp l